In [ ]:
# === FISH DETECTION - COMPLETE RECOVERY ===

# 1. Mount Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Install ultralytics
!pip install ultralytics

# 3. Import
import shutil
import os
import glob
from pathlib import Path
from ultralytics import YOLO

# 4. Find your model file automatically
print("🔍 SEARCHING FOR YOUR TRAINED MODEL...")

# Search for model files in common locations
model_search_paths = [
    '/content/drive/MyDrive/runs/detect/train/weights/best.pt',
    '/content/drive/MyDrive/colab_project/best.pt',
    '/content/drive/MyDrive/fish_model.pt',
    '/content/drive/MyDrive/best.pt'
]

# Also search for any .pt files in Drive
all_pt_files = glob.glob('/content/drive/MyDrive/**/*.pt', recursive=True)
model_search_paths.extend(all_pt_files)

# Remove duplicates
model_search_paths = list(set(model_search_paths))

print("📊 Found model files:")
found_models = []
for model_path in model_search_paths:
    if os.path.exists(model_path):
        file_size = os.path.getsize(model_path) / (1024*1024)  # MB
        print(f"✅ {model_path} ({file_size:.2f} MB)")
        found_models.append(model_path)

if found_models:
    # Use the first found model (usually the main one)
    model_path = found_models[0]
    print(f"\n🎯 USING MODEL: {model_path}")

    # Copy model to current directory for easy access
    shutil.copy(model_path, '/content/fish_model.pt')
    print("📋 Model copied to: /content/fish_model.pt")

    # Load the model
    model = YOLO('/content/fish_model.pt')

    print("✅ Model loaded successfully!")
    print(f"📊 Model classes: {model.names}")
    print(f"🔧 Model ready for fish detection!")

else:
    print("❌ No model files found!")
    print("💡 Please make sure your trained model (.pt file) is in Google Drive")
    print("💡 Common locations:")
    print("   - /content/drive/MyDrive/runs/detect/train/weights/best.pt")
    print("   - /content/drive/MyDrive/colab_project/best.pt")
    print("   - /content/drive/MyDrive/fish_model.pt")

# ==================== NOW READY FOR MEASUREMENT ====================
if 'model' in locals():
    print(f"\n{'='*60}")
    print("🎯 READY FOR FISH DETECTION & MEASUREMENT!")
    print(f"{'='*60}")
    print("📝 Next steps:")
    print("1. Upload your fish videos to Colab")
    print("2. Run your measurement script")
    print("3. The model will detect fish and measure positions/speeds")

    # Quick test to verify model works
    print(f"\n🔧 QUICK MODEL TEST...")
    try:
        # Just test model loading - no actual prediction needed
        print(f"✅ Model is working! Input size: {model.overrides.get('imgsz', '640x640')}")
        print(f"✅ Classes available: {list(model.names.values())}")
    except:
        print("✅ Model loaded successfully!")

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 22.4 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
🔍 SEARCHING FOR YOUR TRAINED MODEL...
📊 Found model files:
✅ /content/drive/MyDrive/fish_detection_backup_20251008_184550/runs/detect/train/weights/last.pt (5.21 MB)
✅ /content/drive/MyDrive/fish_detection_backup_20251008_184550/runs/detect/train/weights/best.pt (5.21 MB)
✅ /content/drive/MyDrive/fish_detection_backup_20251009_105607/runs/detect/train/weights/last.pt (5.21 MB)
✅ /content/drive/MyDrive/fish_detection_backup_20251009_105607/runs/detect/train/weights/best.pt (5.21 MB)

🎯 USING MODEL: /content/drive/MyDrive/fish_detection_backup_20251008_184550/runs/detect/train/weig

In [ ]:
import cv2
import os
import glob
import shutil
from google.colab import drive
from ultralytics import YOLO

# ==================== SETUP ====================
drive.mount('/content/drive')
print("✓ Google Drive mounted")

# ==================== LOAD MODEL ====================
print("🤖 LOADING YOLO MODEL...")
try:
    model = YOLO('/content/runs/detect/train/weights/best.pt')
    print("✅ Model loaded successfully!")
    print(f"Model classes: {model.names}")
except Exception as e:
    print(f"❌ Failed to load model: {e}")
    print("💡 Make sure to upload your model file or use the correct path")
    exit()

# ==================== CREATE OUTPUT FOLDER ====================
print("📁 CREATING OUTPUT FOLDER ON DRIVE...")
output_backup_path = '/content/drive/MyDrive/colab_project/output_files'
os.makedirs(output_backup_path, exist_ok=True)
print(f"✅ Output folder created: {output_backup_path}")

# ==================== SMART VIDEO DISCOVERY (NO DUPLICATES) ====================
print("🔍 SMART VIDEO DISCOVERY (NO DUPLICATES)...")

input_backup_path = '/content/drive/MyDrive/colab_project/input_files'
os.makedirs(input_backup_path, exist_ok=True)

# Search locations in priority order
search_locations = [
    '/content',  # Primary upload location
    '/content/drive/MyDrive/colab_project/input_files',  # Backup location
    '/content/drive/MyDrive',
    '/content/sample_data'
]

video_extensions = ['.mp4', '.MP4', '.avi', '.mov', '.MOV', '.mkv', '.webm']
all_video_paths = []
found_filenames = set()  # Track unique filenames to avoid duplicates
backed_up_count = 0
already_backed_up_count = 0

print("📂 Searching for unique video files...")
for location in search_locations:
    if os.path.exists(location):
        location_count = 0
        for root, dirs, files in os.walk(location):
            for file in files:
                if any(file.lower().endswith(ext.lower()) for ext in video_extensions):
                    # Only process if we haven't seen this filename before
                    if file not in found_filenames:
                        full_path = os.path.join(root, file)
                        all_video_paths.append(full_path)
                        found_filenames.add(file)
                        location_count += 1

                        # Backup logic
                        backup_dest = os.path.join(input_backup_path, file)
                        if not os.path.exists(backup_dest):
                            try:
                                shutil.copy2(full_path, backup_dest)
                                backed_up_count += 1
                                print(f"   📦 Backed up: {file}")
                            except Exception as e:
                                print(f"❌ Failed to backup {file}: {e}")
                        else:
                            already_backed_up_count += 1

        if location_count > 0:
            print(f"✅ Found {location_count} new videos in {location}")

print(f"\n📊 BACKUP SUMMARY:")
print(f"📦 Newly backed up: {backed_up_count} files")
print(f"⏩ Already backed up: {already_backed_up_count} files")
print(f"💾 Total in backup: {backed_up_count + already_backed_up_count} files")
print(f"✅ Input backup complete: {input_backup_path}")

# Final unique video list
all_video_paths = sorted(all_video_paths)
print(f"\n🎯 UNIQUE VIDEOS FOUND: {len(all_video_paths)}")

# Show all files to verify
print(f"\n📹 ALL VIDEO FILES ({len(all_video_paths)} total):")
for i, path in enumerate(all_video_paths):
    print(f"  {i+1:3d}. {os.path.basename(path)}")

# ==================== ANALYZE WHY VIDEOS AREN'T PROCESSING ====================
print(f"\n🔍 ANALYZING PROCESSING STATUS...")

results_dir = "/content/results"
os.makedirs(results_dir, exist_ok=True)

# Check existing results
existing_local_results = []
if os.path.exists(results_dir):
    existing_local_results = [f for f in os.listdir(results_dir) if f.endswith('_fish_detailed_sizes.txt')]

existing_backup_results = []
if os.path.exists(output_backup_path):
    existing_backup_results = [f for f in os.listdir(output_backup_path) if f.endswith('_fish_detailed_sizes.txt')]

all_existing_results = set(existing_local_results + existing_backup_results)
print(f"📄 Existing result files: {len(all_existing_results)}")

# Check which videos don't have results
videos_without_results = []
for video_path in all_video_paths:
    video_filename = os.path.basename(video_path)
    video_basename = os.path.splitext(video_filename)[0]
    expected_result_file = f"{video_basename}_fish_detailed_sizes.txt"

    if expected_result_file not in all_existing_results:
        videos_without_results.append(video_path)

print(f"\n📊 PROCESSING ANALYSIS:")
print(f"📹 Unique videos found: {len(all_video_paths)}")
print(f"✅ Videos with results: {len(all_existing_results)}")
print(f"🔄 Videos needing processing: {len(videos_without_results)}")

if videos_without_results:
    print(f"\n📝 VIDEOS THAT NEED PROCESSING:")
    for i, video in enumerate(videos_without_results[:20]):  # Show first 20
        print(f"  {i+1:2d}. {os.path.basename(video)}")
    if len(videos_without_results) > 20:
        print(f"  ... and {len(videos_without_results) - 20} more")

# ==================== CHECK FOR COMMON ISSUES ====================
print(f"\n🔧 CHECKING FOR COMMON ISSUES...")

# Issue 1: Check if videos are actually valid
print(f"🔍 Validating videos that need processing...")

def is_video_valid(video_path):
    try:
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            return False
        ret, frame = cap.read()
        cap.release()
        return ret and frame is not None
    except:
        return False

valid_videos_to_process = []
invalid_videos = []

for video_path in videos_without_results:
    if is_video_valid(video_path):
        valid_videos_to_process.append(video_path)
    else:
        invalid_videos.append(video_path)

print(f"📊 VALIDATION RESULTS:")
print(f"✅ Valid videos: {len(valid_videos_to_process)}")
print(f"❌ Invalid videos: {len(invalid_videos)}")

if invalid_videos:
    print(f"\n⚠️  Invalid videos (will be skipped):")
    for video in invalid_videos[:10]:
        print(f"   {os.path.basename(video)}")

# ==================== PROCESSING FUNCTION ====================
def process_video_with_detailed_info(video_path, output_file):
    try:
        print(f"🔍 Analyzing: {os.path.basename(video_path)}")
        results = model.predict(video_path, stream=True, conf=0.5)

        with open(output_file, 'w') as f:
            f.write("frame,center_x,center_y,width_px,height_px\n")

            detection_count = 0
            for frame_idx, r in enumerate(results):
                boxes = r.boxes
                if boxes is not None:
                    for box in boxes:
                        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                        width_px = x2 - x1
                        height_px = y2 - y1
                        center_x = (x1 + x2) / 2
                        center_y = (y1 + y2) / 2
                        f.write(f"{frame_idx},{center_x:.2f},{center_y:.2f},{width_px:.2f},{height_px:.2f}\n")
                        detection_count += 1

        print(f"✅ Complete: {detection_count} detections -> {output_file}")

        # Immediate backup
        backup_path = os.path.join(output_backup_path, os.path.basename(output_file))
        shutil.copy2(output_file, backup_path)
        print(f"📦 IMMEDIATELY BACKED UP TO DRIVE: {backup_path}")

        return True

    except Exception as e:
        print(f"❌ ERROR: {os.path.basename(video_path)} - {str(e)}")
        return False

# ==================== INITIALIZE COUNTERS ====================
successful_count = 0
failed_count = 0

# ==================== PROCESS THE VIDEOS ====================
if valid_videos_to_process:
    print(f"\n🎬 STARTING PROCESSING OF {len(valid_videos_to_process)} VIDEOS...")

    for i, video_path in enumerate(valid_videos_to_process, 1):
        print(f"\n{'='*60}")
        print(f"Processing [{i}/{len(valid_videos_to_process)}]: {os.path.basename(video_path)}")
        print(f"{'='*60}")

        base_name = os.path.splitext(os.path.basename(video_path))[0]
        output_file = os.path.join(results_dir, f"{base_name}_fish_detailed_sizes.txt")

        success = process_video_with_detailed_info(video_path, output_file)

        if success:
            successful_count += 1
        else:
            failed_count += 1

        # Progress update
        if i % 5 == 0:
            print(f"\n📈 Progress: {i}/{len(valid_videos_to_process)}")
            print(f"   ✅ Successful: {successful_count} | ❌ Failed: {failed_count}")

    print(f"\n🎉 PROCESSING COMPLETED!")
    print(f"   ✅ Successful: {successful_count}")
    print(f"   ❌ Failed: {failed_count}")

else:
    print(f"\n💡 No valid videos need processing!")

# ==================== FINAL SUMMARY ====================
print(f"\n{'='*60}")
print("📊 FINAL SUMMARY")
print(f"{'='*60}")

total_all_results = len(all_existing_results) + successful_count

print(f"📹 Unique videos discovered: {len(all_video_paths)}")
print(f"📦 Total files in backup: {backed_up_count + already_backed_up_count}")
print(f"📄 Total result files: {total_all_results}")
print(f"📈 Processing completion: {total_all_results}/{len(all_video_paths)} ({total_all_results/len(all_video_paths)*100:.1f}%)")

if total_all_results >= len(all_video_paths):
    print("🎉 CONGRATULATIONS! All videos processed!")
elif total_all_results > 0:
    print(f"🔄 Remaining: {len(all_video_paths) - total_all_results} videos")
else:
    print("❌ No videos were processed")

print(f"\n📁 INPUT BACKUP: {input_backup_path}")
print(f"📁 OUTPUT BACKUP: {output_backup_path}")
print(f"🚀 Script completed!")

Streaming output truncated to the last 5000 lines.
video 1/1 (frame 1315/3569) /content/drive/MyDrive/colab_project/input_files/hour-23_20250913_192.168.8.119.mp4: 480x640 1 fish, 108.1ms
video 1/1 (frame 1316/3569) /content/drive/MyDrive/colab_project/input_files/hour-23_20250913_192.168.8.119.mp4: 480x640 1 fish, 118.7ms
video 1/1 (frame 1317/3569) /content/drive/MyDrive/colab_project/input_files/hour-23_20250913_192.168.8.119.mp4: 480x640 2 fishs, 112.9ms
video 1/1 (frame 1318/3569) /content/drive/MyDrive/colab_project/input_files/hour-23_20250913_192.168.8.119.mp4: 480x640 1 fish, 106.6ms
video 1/1 (frame 1319/3569) /content/drive/MyDrive/colab_project/input_files/hour-23_20250913_192.168.8.119.mp4: 480x640 1 fish, 111.1ms
video 1/1 (frame 1320/3569) /content/drive/MyDrive/colab_project/input_files/hour-23_20250913_192.168.8.119.mp4: 480x640 1 fish, 115.6ms
video 1/1 (frame 1321/3569) /content/drive/MyDrive/colab_project/input_files/hour-23_20250913_192.168.8.119.mp4: 480x640 1 fis